#### 1. CRIAÇÃO DAS TABELAS DIMENSÕES 

In [3]:
from pathlib import Path
import pandas as pd

PROCESSED_PATH = Path("../data/processed")

df = pd.read_parquet(
    PROCESSED_PATH / "exportacoes.parquet"
)

# ==========================================
# PADRONIZAÇÃO
# ==========================================

for coluna in ["co_pais", "co_ncm", "sg_uf_ncm"]:
    df[coluna] = df[coluna].astype(str)

# ==========================================
# DIMENSÃO TEMPO
# ==========================================

dim_tempo = (
    df[["co_ano", "co_mes"]]
    .drop_duplicates()
    .sort_values(["co_ano", "co_mes"])
    .reset_index(drop=True)
)

dim_tempo["trimestre"] = (
    ((dim_tempo["co_mes"] - 1) // 3) + 1
)

dim_tempo["id_tempo"] = (
    dim_tempo.index + 1
)

# ==========================================
# DIMENSÃO ESTADO
# ==========================================

dim_estado = pd.DataFrame({
    "sg_uf_ncm": [
        "AC","AL","AP","AM","BA","CE","DF","ES",
        "GO","MA","MT","MS","MG","PA","PB","PR",
        "PE","PI","RJ","RN","RS","RO","RR","SC",
        "SP","SE","TO"
    ],
    "estado": [
        "Acre","Alagoas","Amapá","Amazonas","Bahia",
        "Ceará","Distrito Federal","Espírito Santo",
        "Goiás","Maranhão","Mato Grosso",
        "Mato Grosso do Sul","Minas Gerais",
        "Pará","Paraíba","Paraná","Pernambuco",
        "Piauí","Rio de Janeiro",
        "Rio Grande do Norte",
        "Rio Grande do Sul",
        "Rondônia","Roraima",
        "Santa Catarina",
        "São Paulo","Sergipe","Tocantins"
    ],
    "regiao": [
        "Norte","Nordeste","Norte","Norte","Nordeste",
        "Nordeste","Centro-Oeste","Sudeste",
        "Centro-Oeste","Nordeste","Centro-Oeste",
        "Centro-Oeste","Sudeste",
        "Norte","Nordeste","Sul","Nordeste",
        "Nordeste","Sudeste","Nordeste",
        "Sul","Norte","Norte",
        "Sul","Sudeste","Nordeste","Norte"
    ]
})

dim_estado["id_estado"] = (
    dim_estado.index + 1
)

# ==========================================
# DIMENSÃO PAÍS
# ==========================================

dim_pais = (
    df[["co_pais"]]
    .drop_duplicates()
    .sort_values("co_pais")
    .reset_index(drop=True)
)

dim_pais["id_pais"] = (
    dim_pais.index + 1
)

# ==========================================
# DIMENSÃO NCM
# ==========================================

dim_ncm = (
    df[["co_ncm"]]
    .drop_duplicates()
    .sort_values("co_ncm")
    .reset_index(drop=True)
)

dim_ncm["id_ncm"] = (
    dim_ncm.index + 1
)

#### 2. CRIAÇÃO DA TABELA FATO 

In [4]:
fact_exportacao = (
    df
    .merge(
        dim_tempo[
            ["co_ano", "co_mes", "id_tempo"]
        ],
        on=["co_ano", "co_mes"],
        how="left"
    )
    .merge(
        dim_estado[
            ["sg_uf_ncm", "id_estado"]
        ],
        on="sg_uf_ncm",
        how="left"
    )
    .merge(
        dim_pais[
            ["co_pais", "id_pais"]
        ],
        on="co_pais",
        how="left"
    )
    .merge(
        dim_ncm[
            ["co_ncm", "id_ncm"]
        ],
        on="co_ncm",
        how="left"
    )
)

fact_exportacao = fact_exportacao[
    [
        "id_tempo",
        "id_estado",
        "id_pais",
        "id_ncm",
        "vl_fob",
        "kg_liquido",
        "valor_medio_kg"
    ]
]

fact_exportacao.head()

,id_tempo,id_estado,id_pais,id_ncm,vl_fob,kg_liquido,valor_medio_kg
0,3,19.0,18,6313,20,5,4.0
1,12,25.0,204,5720,84,0,NaN
2,4,21.0,92,6457,758,2,379.0
3,5,19.0,163,7604,275,1,275.0
4,10,13.0,163,7203,9,0,NaN


#### 3. EXPORTAÇÃO DAS TABELAS AUXILIARES

In [12]:
BASE_PATH = Path("../data")

df = pd.read_parquet(
    BASE_PATH / "processed" / "exportacoes.parquet"
)

dim_pais_aux = pd.read_csv(
    BASE_PATH / "auxiliar" / "PAIS.csv",
    sep=";",
    encoding="latin1"
)

dim_ncm_aux = pd.read_csv(
    BASE_PATH / "auxiliar" / "NCM.csv",
    sep=";",
    encoding="latin1"
)

dim_estado_aux = pd.read_csv(
    BASE_PATH / "auxiliar" / "UF.csv",
    sep=";",
    encoding="latin1"
)

#### 4. ENREQUECIMENTO DOS DADOS E CRIAÇÃO DAS DIMENSÕES FINAIS

In [14]:
# ==========================================================
# ENRIQUECIMENTO DOS DADOS E CRIAÇÃO DAS DIMENSÕES
# ==========================================================

# Garantir compatibilidade dos códigos para merge

df["co_pais"] = df["co_pais"].astype(int)
df["co_ncm"] = df["co_ncm"].astype(str)

dim_pais_aux["CO_PAIS"] = dim_pais_aux["CO_PAIS"].astype(int)
dim_ncm_aux["CO_NCM"] = dim_ncm_aux["CO_NCM"].astype(str)

# ----------------------------------------------------------
# DIMENSÃO TEMPO
# ----------------------------------------------------------

dim_tempo = (
    df[["co_ano", "co_mes"]]
    .drop_duplicates()
    .sort_values(["co_ano", "co_mes"])
    .reset_index(drop=True)
)

dim_tempo["trimestre"] = (
    ((dim_tempo["co_mes"] - 1) // 3) + 1
)

dim_tempo["id_tempo"] = dim_tempo.index + 1

# ----------------------------------------------------------
# DIMENSÃO PAÍS
# ----------------------------------------------------------

dim_pais = (
    df[["co_pais"]]
    .drop_duplicates()
    .merge(
        dim_pais_aux[
            ["CO_PAIS", "NO_PAIS"]
        ],
        left_on="co_pais",
        right_on="CO_PAIS",
        how="left"
    )
    .drop(columns="CO_PAIS")
    .rename(
        columns={
            "NO_PAIS": "nome_pais"
        }
    )
    .sort_values("co_pais")
    .reset_index(drop=True)
)

dim_pais["id_pais"] = dim_pais.index + 1

# ----------------------------------------------------------
# DIMENSÃO NCM
# ----------------------------------------------------------

dim_ncm = (
    df[["co_ncm"]]
    .drop_duplicates()
    .merge(
        dim_ncm_aux[
            ["CO_NCM", "NO_NCM_POR"]
        ],
        left_on="co_ncm",
        right_on="CO_NCM",
        how="left"
    )
    .drop(columns="CO_NCM")
    .rename(
        columns={
            "NO_NCM_POR": "descricao_produto"
        }
    )
    .sort_values("co_ncm")
    .reset_index(drop=True)
)

dim_ncm["id_ncm"] = dim_ncm.index + 1

# ----------------------------------------------------------
# DIMENSÃO ESTADO
# ----------------------------------------------------------

dim_estado = (
    dim_estado_aux[
        ["SG_UF", "NO_UF", "NO_REGIAO"]
    ]
    .rename(
        columns={
            "SG_UF": "sg_uf_ncm",
            "NO_UF": "estado",
            "NO_REGIAO": "regiao"
        }
    )
    .reset_index(drop=True)
)

dim_estado["id_estado"] = dim_estado.index + 1

print("Dimensões criadas com sucesso!")

Dimensões criadas com sucesso!


#### 5. TRANSFORMAÇÃO E CARREGAMENTO DOS ARQUIVOS PARQUET

In [15]:
dim_tempo.to_parquet(
    BASE_PATH / "processed" / "dim_tempo.parquet",
    index=False
)

dim_pais.to_parquet(
    BASE_PATH / "processed" / "dim_pais.parquet",
    index=False
)

dim_ncm.to_parquet(
    BASE_PATH / "processed" / "dim_ncm.parquet",
    index=False
)

dim_estado.to_parquet(
    BASE_PATH / "processed" / "dim_estado.parquet",
    index=False
)

fact_exportacao.to_parquet(
    BASE_PATH / "processed" / "fact_exportacao.parquet",
    index=False
)

print("Arquivos exportados com sucesso.")

Arquivos exportados com sucesso.


#### 6. VALIDAÇÃO DAS TABELAS

In [16]:
print(dim_pais.head())
print(dim_ncm.head())
print(dim_estado.head())
print(fact_exportacao.head())

   co_pais     nome_pais  id_pais
0       13   Afeganistão        1
1       15  Aland, Ilhas        2
2       17       Albânia        3
3       23      Alemanha        4
4       31  Burkina Faso        5
     co_ncm                                  descricao_produto  id_ncm
0  10011100                         Trigo duro, para semeadura       1
1  10011900                  Trigo duro, exceto para semeadura       2
2  10019100  Outrros trigos e misturas de trigo com centeio...       3
3  10019900  Outros trigos e misturas de trigo com centeio,...       4
4  10021000                            Centeio, para semeadura       5
  sg_uf_ncm              estado                regiao  id_estado
0        ZN  Zona Não Declarada  REGIAO NAO DECLARADA          1
1        RO            Rondônia          REGIAO NORTE          2
2        AC                Acre          REGIAO NORTE          3
3        AM            Amazonas          REGIAO NORTE          4
4        RR             Roraima          REGI